Notebook to validate if platform models sent predictions for all geocodes and validation sets per challenge.

In [1]:
import numpy as np
import pandas as pd
import mosqlient as mosq
from epiweeks import Week

In [2]:
import os
from dotenv import load_dotenv

# Access the environment variables
api_key = os.getenv('api_key')

In [3]:
def validate_preds(df_preds_all, disease='A90', col='adm_1'): 
    # 1. Filtrar pela doença selecionada (.copy() evita avisos de SettingWithCopyWarning)
    df_preds_info = df_preds_all.loc[df_preds_all.disease == disease].copy()
    df_preds_info = df_preds_info.loc[df_preds_info.adm_1 != 32]
    df_preds_info = df_preds_info.loc[df_preds_info.start < Week(2026, 41).startdate()]

    # 2. CORREÇÃO: Aplicar a exceção do adm_1 == 32 se estivermos analisando estados
    if col == 'adm_1':
        df_preds_info = df_preds_info.loc[(df_preds_info.adm_2.isna())]

    # 3. CORREÇÃO: Mapeamento de regras seguro (evita UnboundLocalError e remove o '&')
    valid_configs = {
        ('adm_1', 'A90'): {'size': 26, 'challenge': 'dengue_state'},
        ('adm_1', 'A92.0'): {'size': 26, 'challenge': 'chik_state'},
        ('adm_2', 'A90'): {'size': 15, 'challenge': 'dengue_city'},
        ('adm_2', 'A92.0'): {'size': 10, 'challenge': 'chik_city'},
    }

    # Valida se os parâmetros passados existem nas regras de negócio
    if (col, disease) not in valid_configs:
        raise ValueError(f"A combinação de coluna '{col}' e doença '{disease}' não é válida ou configurada.")

    # Resgata os valores corretos com segurança
    config = valid_configs[(col, disease)]
    size = config['size']
    challenge = config['challenge']

    # 4. Agrupamento detalhado por modelo, doença e localidade
    grupo_detalhado = (
        df_preds_info.groupby(['model_name', 'disease', col])
        .size()
        .reset_index(name='qtd_validacoes')
    )

    # 5. Agrupamento final para gerar o relatório
    relatorio_validacao = (
        grupo_detalhado.groupby(['model_name', 'disease'])
        .agg(
            qtd_adm=(col, 'nunique'),
            todos_com_4_validaçoes=('qtd_validacoes', lambda x: (x == 4).all())
        )
        .reset_index()
    )

    # Trata o caso de o retorno ser um DataFrame vazio para não quebrar a lógica
    if relatorio_validacao.empty:
        return pd.DataFrame(columns=['model_name', 'disease', 'qtd_adm', 'todos_com_4_validaçoes', 'challenge', 'dados_corretos'])

    # 6. Adiciona as colunas de validação final
    relatorio_validacao['challenge'] = challenge
    relatorio_validacao['dados_corretos'] = (
        (relatorio_validacao['qtd_adm'] == size) & 
        (relatorio_validacao['todos_com_4_validaçoes'])
    )

    return relatorio_validacao

def load_preds(api_key = api_key, model_name = 'str'): 

    preds = mosq.get_predictions(api_key = api_key, model_name= model_name)

    if len(preds) >0 :
        df_info = pd.DataFrame([
            {   "pred": pred, 
                "model_name": pred.model.repository,
                "id": pred.id, 
                "commit": pred.commit, 
                "disease": pred.disease,
                "adm_1": pred.adm_1,
                "adm_2": pred.adm_2, 
                "start": pred.start,
                "end": pred.end,
                "wis": pred.scores.get("wis")
                if pred.scores is not None else None,
                "published": pred.published }
            for pred in preds
        ])

        df_info = df_info.loc[df_info.published == True]

        df_info['validation'] = None 

        df_info.loc[(df_info.start == Week(2022, 41).startdate()) & (df_info.end == Week(2023, 40).startdate()), 'validation'] = 1 
        df_info.loc[(df_info.start == Week(2023, 41).startdate()) & (df_info.end == Week(2024, 40).startdate()), 'validation'] = 2 
        df_info.loc[(df_info.start == Week(2024, 41).startdate()) & (df_info.end == Week(2025, 40).startdate()), 'validation'] = 3 
        df_info.loc[(df_info.start == Week(2025, 41).startdate()) & (df_info.end == Week(2026, 40).startdate()), 'validation'] = 4 

        return df_info 


    else: 
        print(f'Nenhum previsão registrada para o modelo {model_name}')





Load 3rd IMDC models: 

In [4]:
models = mosq.get_models(api_key = api_key, imdc_year = 2026)

models

[blaiate/3rd_imdc_-unifesp-_-4mosqueteiras-,
 americocunhajr/3rd_imdc_lncc_clidengo26chikungunya,
 americocunhajr/3rd_imdc_lncc_clidengo26dengue,
 eduardocorrearaujo/3rd_imdc_emap_example,
 DiogoParreira/3rd_imdc_rki_rki_zki_ph,
 DiogoParreira/3rd_imdc_rki_rki_zki_ph_lstm_geo,
 joel-da-silva-cavalcanti-filho/3rd_imdc_unesp_recogna,
 jrjoaorenato/3rd_imdc_unesp_recogna_TTM,
 Ricafya/3rd_imdc_afya_ric,
 lsbastos/3rd_imdc_procc_bb_model,
 marciomacielbastos/3rd_imdc_fgv_sakhal,
 haridas-das/DS-OKSTATE-2026,
 scrocha/3rd_imdc_emap_xgbsillas,
 kamrul28890/3rd_imdc_purdue_neuralearth,
 EzequielEBS/3rd_imdc_emap_epidematicos_prophet,
 Luizsrs/3rd_imdc_fiocruz_zerolags,
 Dududidicao99/3rd_imdc_pucrio_arbocaster,
 DiogoParreira/3rd_imdc_rki_rki_zki_ph_chronos,
 asgouveiaa/3rd_imdc_fiocruz_mard,
 ZuilhoSe/3rd_imdc_fgv_pattern-blue,
 germanavila09/3rd_imdc_universidad_del_valle_grupo_modelamiento_datos_dengue,
 InfraMIND-models/3rd_imdc_ifgw_inframind-proteus,
 SungmokJung/3rd_imdc_nus_nus-cerm,


In [5]:
list_df_val = []

for model in models: 
    
    df_preds_info = load_preds(api_key=api_key, model_name=model.repository.split('/')[1])

    if df_preds_info is not None: 

        list_df_val.append(validate_preds(df_preds_info, disease = 'A90', col = 'adm_1'))
            
        list_df_val.append(validate_preds(df_preds_info, disease = 'A92.0', col = 'adm_1'))

        list_df_val.append(validate_preds(df_preds_info, disease = 'A90', col = 'adm_2'))

        list_df_val.append(validate_preds(df_preds_info, disease = 'A92.0', col = 'adm_2'))
    
df_val = pd.concat(list_df_val, ignore_index= True)
df_val.head()

Nenhum previsão registrada para o modelo 3rd_imdc_unesp_recogna_TTM


100%|██████████| 1/1 [00:00<00:00,  1.50requests/s]


Nenhum previsão registrada para o modelo 3rd_imdc_rki_rki_zki_ph_chronos


100%|██████████| 1/1 [00:00<00:00,  1.64requests/s]


Nenhum previsão registrada para o modelo ADCaptura-container


,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos
0,blaiate/3rd_imdc_-unifesp-_-4mosqueteiras-,A90,26,True,dengue_state,True
1,americocunhajr/3rd_imdc_lncc_clidengo26chikung...,A92.0,26,True,chik_state,True
2,americocunhajr/3rd_imdc_lncc_clidengo26dengue,A90,26,True,dengue_state,True
3,eduardocorrearaujo/3rd_imdc_emap_example,A90,3,False,dengue_state,False
4,DiogoParreira/3rd_imdc_rki_rki_zki_ph,A90,26,True,dengue_state,True


In [13]:
df_preds_info = load_preds(api_key=api_key, model_name='3rd_imdc_lncc_surge_model26_chikungunya')

list_df_val.append(validate_preds(df_preds_info, disease = 'A92.0', col = 'adm_1'))

df_val = pd.concat(list_df_val, ignore_index= True)

df_val.loc[df_val.model_name == 'pesquefGH/3rd_imdc_lncc_surge_model26_dengue_chikungunya']

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos


In [14]:
validate_preds(df_preds_info, disease = 'A92.0', col = 'adm_1')

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos
0,pesquefGH/3rd_imdc_lncc_surge_model26_chikungunya,A92.0,26,True,chik_state,True


In [15]:
df_val.loc[(df_val.dados_corretos == True)  & (df_val.challenge == 'chik_state')].model_name.unique().shape

(11,)

In [17]:
df_val.loc[(df_val.dados_corretos == True)  & (df_val.challenge == 'chik_state')]

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos
1,americocunhajr/3rd_imdc_lncc_clidengo26chikung...,A92.0,26,True,chik_state,True
11,lsbastos/3rd_imdc_procc_bb_model,A92.0,26,True,chik_state,True
14,haridas-das/DS-OKSTATE-2026,A92.0,26,True,chik_state,True
16,scrocha/3rd_imdc_emap_xgbsillas,A92.0,26,True,chik_state,True
20,kamrul28890/3rd_imdc_purdue_neuralearth,A92.0,26,True,chik_state,True
24,EzequielEBS/3rd_imdc_emap_epidematicos_prophet,A92.0,26,True,chik_state,True
36,SungmokJung/3rd_imdc_nus_nus-cerm,A92.0,26,True,chik_state,True
39,pesquefGH/3rd_imdc_lncc_ARp26_chikungunya,A92.0,26,True,chik_state,True
52,EzequielEBS/3rd_imdc_emap_epidematicos_sarimax...,A92.0,26,True,chik_state,True
57,EzequielEBS/3rd_imdc_emap_epidematicos_prophet,A92.0,26,True,chik_state,True


In [7]:
df_val.loc[df_val.model_name == 'pesquefGH/3rd_imdc_lncc_surge_model26_dengue_chikungunya']

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos


In [16]:
df_val.to_csv('predictions/validate_models.csv', index = False)

In [7]:
df_val.loc[df_val.dados_corretos == False]

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos
3,eduardocorrearaujo/3rd_imdc_emap_example,A90,3,False,dengue_state,False
47,felipe-matsuoka123/Dasa_IMDC2026,A90,26,False,dengue_state,False


In [1]:
df_val['author'] = df_val['model_name'].str.split('/').str[0]

df_val.loc[df_val.dados_corretos == True]

NameError: name 'df_val' is not defined

In [11]:
df_val.loc[(df_val.dados_corretos == True) & (df_val.challenge == 'dengue_state')]['author'].nunique()

23

In [15]:
df_val.loc[(df_val.dados_corretos == True) & (df_val.challenge == 'dengue_state')]['author'].unique()

array(['americocunhajr', 'DiogoParreira',
       'joel-da-silva-cavalcanti-filho', 'Ricafya', 'lsbastos',
       'marciomacielbastos', 'haridas-das', 'scrocha', 'kamrul28890',
       'EzequielEBS', 'Luizsrs', 'Dududidicao99', 'asgouveiaa',
       'ZuilhoSe', 'germanavila09', 'InfraMIND-models', 'SungmokJung',
       'pesquefGH', 'mattiamazzoli', 'dievillano', 'graeme-dor',
       'BentoLab-DiseaseDynamics', 'eduardocorrearaujo'], dtype=object)

In [14]:
df_val.loc[(df_val.dados_corretos == True) & (df_val.challenge == 'dengue_state')].drop_duplicates().shape

(26, 7)

In [4]:
code_to_state = {33: 'RJ', 32: 'ES', 41: 'PR', 23: 'CE', 21: 'MA',
 31: 'MG', 42: 'SC', 26: 'PE', 25: 'PB', 24: 'RN', 22: 'PI', 27: 'AL',
 28: 'SE', 35: 'SP', 43: 'RS', 15: 'PA', 16: 'AP', 14: 'RR',  11: 'RO',
 13: 'AM', 12: 'AC', 51: 'MT', 50: 'MS', 52: 'GO', 17: 'TO', 53: 'DF',
 29: 'BA'}

In [5]:
df_preds_info = load_preds(api_key=api_key, model_name='3rd_imdc_lncc_surge_model26_chikungunya')

df_preds_info.head()

,pred,model_name,id,commit,disease,adm_1,adm_2,start,end,wis,published,validation
0,"{""id"":9199,""model"":{""id"":70,""repository"":""pesq...",pesquefGH/3rd_imdc_lncc_surge_model26_chikungunya,9199,0a5ecae63b4c536d67b45771123d8bb6d40176ba,A92.0,12,None,2022-10-09,2023-10-01,2.54,True,1
1,"{""id"":9200,""model"":{""id"":70,""repository"":""pesq...",pesquefGH/3rd_imdc_lncc_surge_model26_chikungunya,9200,0a5ecae63b4c536d67b45771123d8bb6d40176ba,A92.0,27,None,2022-10-09,2023-10-01,36.16,True,1
2,"{""id"":9201,""model"":{""id"":70,""repository"":""pesq...",pesquefGH/3rd_imdc_lncc_surge_model26_chikungunya,9201,0a5ecae63b4c536d67b45771123d8bb6d40176ba,A92.0,13,None,2022-10-09,2023-10-01,2.00,True,1
3,"{""id"":9202,""model"":{""id"":70,""repository"":""pesq...",pesquefGH/3rd_imdc_lncc_surge_model26_chikungunya,9202,0a5ecae63b4c536d67b45771123d8bb6d40176ba,A92.0,16,None,2022-10-09,2023-10-01,3.08,True,1
4,"{""id"":9203,""model"":{""id"":70,""repository"":""pesq...",pesquefGH/3rd_imdc_lncc_surge_model26_chikungunya,9203,0a5ecae63b4c536d67b45771123d8bb6d40176ba,A92.0,29,None,2022-10-09,2023-10-01,392.75,True,1


In [7]:
df_val = pd.read_csv('predictions/validate_models.csv')

df_val.head()

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos
0,blaiate/3rd_imdc_-unifesp-_-4mosqueteiras-,A90,26,True,dengue_state,True
1,americocunhajr/3rd_imdc_lncc_clidengo26chikung...,A92.0,26,True,chik_state,True
2,americocunhajr/3rd_imdc_lncc_clidengo26dengue,A90,26,True,dengue_state,True
3,eduardocorrearaujo/3rd_imdc_emap_example,A90,3,False,dengue_state,False
4,DiogoParreira/3rd_imdc_rki_rki_zki_ph,A90,26,True,dengue_state,True


In [8]:
df_val.loc[df_val.model_name == 'pesquefGH/3rd_imdc_lncc_surge_model26_chikungunya']

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos


In [6]:
validate_preds(df_preds_info, disease = 'A92.0', col = 'adm_1')

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos
0,pesquefGH/3rd_imdc_lncc_surge_model26_chikungunya,A92.0,26,True,chik_state,True


In [38]:
print(len(df_preds_info.loc[df_preds_info.validation == 1].adm_1.unique()))

print(np.setdiff1d(np.array(list(code_to_state)), df_preds_info.loc[df_preds_info.validation == 1].adm_1.unique()))

24
[13 21 32]
